# 개별종목 조합A — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합A 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합A의 피처 값만 지정합니다.
import json

COMBINATION = 'A'
FEATURE_COLUMNS = (
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'macd_hist_ratio',
    'bb_bandwidth',
    'bb_position',
    'atr_ratio',
    'hv_20',
    'vol_ratio_20',
    'obv_slope_20',
    'daily_return',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합A 피처: ('sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'macd_hist_ratio', 'bb_bandwidth', 'bb_position', 'atr_ratio', 'hv_20', 'vol_ratio_20', 'obv_slope_20', 'daily_return', 'five_day_return')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4817,0.5012,-0.0195,0.3353,0.3629,0.0687,0.3768,0.1201,0.2241
1,2,balanced,980,20150123,20150421,0.3856,0.3978,-0.0122,0.3647,0.3692,0.0583,0.3802,0.2875,0.3404
2,3,balanced,1210,20151228,20160328,0.3628,0.3762,-0.0134,0.3629,0.3649,0.0497,0.3784,0.3731,0.3662
3,4,balanced,1439,20161202,20170228,0.4515,0.4617,-0.0102,0.3752,0.3853,0.0955,0.4011,0.1934,0.2985
4,5,balanced,1669,20171113,20180207,0.4107,0.3901,0.0206,0.3875,0.3927,0.0960,0.3944,0.3256,0.3710
5,6,balanced,1899,20181024,20190118,0.4056,0.3725,0.0331,0.4038,0.4134,0.1247,0.4246,0.4877,0.4290
6,7,balanced,2129,20190930,20191224,0.4525,0.4781,-0.0256,0.3604,0.3771,0.0858,0.3972,0.2529,0.3356
7,8,balanced,2359,20200902,20201130,0.3793,0.3476,0.0316,0.3792,0.3867,0.0809,0.3960,0.4728,0.4060
8,9,balanced,2589,20210806,20211105,0.3897,0.3914,-0.0017,0.3800,0.3916,0.0814,0.3862,0.2867,0.3454
9,10,balanced,2818,20220714,20221012,0.3716,0.3454,0.0262,0.3709,0.3772,0.0640,0.3814,0.2924,0.3407


,OOS 폴드 평균
accuracy,0.4071
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0103
macro_f1,0.3757
balanced_accuracy,0.3841
mcc,0.0827
pr_auc_macro_ovr,0.3926
down_recall,0.3282
core_harmonic_mean,0.3555


재실행 명령: python scripts/run_stock_model_experiment.py
